# NexusAI Forecast — Exploration Notebook
Author: Avinash Krishna — Team AVV Elites (SIH26153)

SIH26153 · Quick-look EDA on the traffic dataset and a walk through one forecast end to end.
Run `python -m src.train` from the repo root before the later cells (they load the checkpoints).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
import pandas as pd, numpy as np
from src.synthetic_data import generate_synthetic_dataset
from src.features.windowing import FUSED_FEATURE_NAMES

df = generate_synthetic_dataset()
df.describe()

In [ ]:
df['label'].value_counts().plot(kind='bar', title='Window label distribution')

## Feature separation by stage
Sanity check: do the engineered features actually separate the five kill-chain stages? (They should — this drives both the rule-based mapper and the classifier.)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, ['port_scan_score', 'bytes_per_sec', 'fwd_bwd_byte_ratio']):
    df.boxplot(column=feat, by='label', ax=ax, rot=45)
    ax.set_title(feat)
plt.suptitle('')
plt.tight_layout()

## Load trained checkpoints and run one forecast
Requires `python -m src.train` to have been run first.

In [ ]:
import torch
from src.utils import load_config, resolve_path
from src.models.state_encoder import StateEncoder
from src.models.transition_model import TransitionModel
from src.models.rollout import k_step_rollout
from src.features.windowing import build_sequences

cfg = load_config()
ckpt_dir = resolve_path(cfg['paths']['checkpoint_dir'])
encoder = StateEncoder(cfg); encoder.load_state_dict(torch.load(ckpt_dir / 'state_encoder.pt', map_location='cpu')); encoder.eval()
transition = TransitionModel(cfg); transition.load_state_dict(torch.load(ckpt_dir / 'transition_model.pt', map_location='cpu')); transition.eval()
norm = np.load(ckpt_dir / 'normalization.npz')

df_norm = df.copy()
df_norm[FUSED_FEATURE_NAMES] = (df[FUSED_FEATURE_NAMES].to_numpy(dtype=np.float32) - norm['mean']) / norm['std']
X_seq, y_labels, host_ids = build_sequences(df_norm, sequence_length=cfg['windowing']['sequence_length'])
print(f'{len(X_seq)} sequences built across {len(set(host_ids))} hosts')

In [ ]:
idx = 0
sample = torch.tensor(X_seq[idx:idx+1], dtype=torch.float32)
with torch.no_grad():
    state_seq = encoder.encode_sequence(sample)
result = k_step_rollout(transition, state_seq, k_steps=cfg['rollout']['k_steps'], horizon_minutes=cfg['rollout']['horizon_minutes'])
print('Host:', host_ids[idx], '| True label:', y_labels[idx])
print('Infiltration probability by horizon:', dict(zip(result.horizon_minutes, [round(p,3) for p in result.infiltration_probs])))